# Data Retrieval Bluesky

| Field | Details |
|---|---|
| **Time window** | 5 Jul 2024 – 4 Nov 2024 |
| **Source** | Bluesky Social — `atproto` Python client |
| **Collection** | Paginated hashtag search via API (62 election hashtags) |
| **Subreddits** | n/a — Bluesky platform only |
| **Method** | Search each hashtag; fetch reply threads; filter inline to date window and election keywords (same keyword set as Reddit) |
| **Keywords** | trump · harris · kamala · biden · vance · walz · election · vote · ballot · debate · maga · gop · dnc · rnc · swing state · battleground · project2025 · … |
| **Saved to** | `Data/1_Bronze/Bluesky/bsky_US_2024_raw.csv` |

### bsky_US_2024_raw.csv 32,866 rows × 13 columns

| Column | Type | Description |
|---|---|---|
| `uri` | str | Unique Bluesky post URI |
| `author` | str | Bluesky handle of the poster |
| `display` | str | Display name of the poster |
| `text` | str | Post text (URLs stripped) |
| `timestamp` | str | ISO 8601 creation time (UTC) |
| `likes` | int | Like count at collection time |
| `reposts` | int | Repost count at collection time |
| `replies` | int | Reply count at collection time |
| `mentions` | list | List of @-mentioned handles |
| `is_reply` | bool | True if this is a reply to another post |
| `post_type` | str | `post` (top-level) or `comment` (reply) |
| `query` | str | Hashtag query that retrieved the post, or `thread` for fetched comments |
| `parent_uri` | str | URI of the direct parent post/comment (`None` for top-level posts) |

<!-- toc -->
## Contents
- [Setup](#setup)
- [1. Search Parameters](#1-search-parameters)
- [2. Helper Functions](#2-helper-functions)
- [3. Collect and Save](#3-collect-and-save)
  - [bsky_US_2024_raw.csv columns](#bsky_us_2024_rawcsv-columns)


## Setup

In [ ]:
import sys, os, re, time, warnings
import pandas as pd
warnings.filterwarnings("ignore")

from atproto import Client as BskyClient

config_path = os.path.join("..", "A. Lectures", "Lecture 1 Social network analysis")
sys.path.insert(0, config_path)
from bsky_config_students import BLUESKY_USERNAME, BLUESKY_APP_PASSWORD

client = BskyClient()
client.login(BLUESKY_USERNAME, BLUESKY_APP_PASSWORD)
print(f"Authenticated as: {BLUESKY_USERNAME}")

## 1. Search Parameters

In [ ]:
# Time window
DATE_SINCE = "2024-07-05T00:00:00Z"
DATE_UNTIL = "2024-11-04T23:59:59Z"

# Election keywords — identical to Reddit keyword list
ELECTION_KEYWORDS = [
    # Candidates & running mates
    'trump', 'donald trump', 'harris', 'kamala', 'kamala harris',
    'biden', 'joe biden', 'vance', 'jd vance', 'walz', 'tim walz',

    # Election process
    'election', 'election2024', 'us election', 'electionday',
    'vote', 'vote2024', 'voting', 'voter', 'voterregistration',
    'ballot', 'earlyvoting', 'mail-in voting', 'mailinvoting',
    'poll', 'polls', 'electoral', 'swing state', 'battleground',
    'electionintegrity', 'america decides', 'decision2024',

    # Campaign & parties
    'campaign', 'debate', 'presidential debate', 'vp debate',
    'trumpdebate', 'trumpharrisdebate', 'debatenight',
    'president', 'presidential',
    'democrat', 'democratic', 'democrats',
    'republican', 'republicans',
    'maga', 'maga2024', 'america first', 'americafirst',
    'gop', 'dnc', 'rnc',
    'project2025',

    # Conventions & events
    'convention', 'rnc2024', 'dnc2024', 'republican convention',
    'democratic convention',
    'trump rally', 'trumprally',
    'inauguration', 'primary',

    # Biden exit
    'biden drops out', 'bidendropsout', 'biden out', 'biden withdraws',

    # Slogans
    'vote blue', 'voteblue', 'vote trump', 'votetrump',
    'vote harris', 'voteharris', 'vote kamala',
    'we are not going back', 'harriswalz', 'trumpvance',
]

# Hashtags used as API search queries (62 election hashtags)
HASHTAGS = [
    # Core election
    "#Election2024", "#USElection2024", "#ElectionDay", "#Vote2024",
    "#Presidential2024", "#PresidentialElection", "#ElectionNight",
    "#AmericaDecides", "#Decision2024", "#VoterRegistration",
    "#EarlyVoting", "#MailInVoting", "#ElectionIntegrity",
    "#Battleground2024", "#SwingState",

    # Trump / Republican
    "#Trump2024", "#TrumpVance", "#VoteTrump", "#Trump",
    "#DonaldTrump", "#MAGA", "#MAGA2024", "#AmericaFirst",
    "#TrumpRally", "#Republicans", "#GOP", "#Republican",
    "#JDVance", "#Vance2024", "#VanceVP",
    "#RNC2024", "#RepublicanConvention",
    "#Project2025", "#TrumpDebate",

    # Harris / Democrat
    "#Harris2024", "#KamalaHarris2024", "#KamalaHarris",
    "#HarrisWalz", "#VoteHarris", "#VoteBlue", "#VoteKamala",
    "#Kamala2024", "#Kamala", "#Harris",
    "#TimWalz", "#Walz2024", "#WalzVP",
    "#DNC2024", "#DemConvention", "#DemocraticConvention",
    "#WeAreNotGoingBack", "#WinWithKamala",
    "#Democrats", "#Democrat",

    # Biden exit
    "#BidenDropsOut", "#BidenOut", "#BidenWithdraws",

    # Debates & key moments
    "#PresidentialDebate", "#Debate2024", "#VPDebate",
    "#TrumpHarrisDebate", "#DebateNight",
]

# Output path
BRONZE_CSV = "../Data/1_Bronze/Bluesky/bsky_US_2024_raw.csv"

print(f"Window   : {DATE_SINCE[:10]}  to  {DATE_UNTIL[:10]}")
print(f"Hashtags : {len(HASHTAGS)}")
print(f"Keywords : {len(ELECTION_KEYWORDS)}")

## 2. Helper Functions

In [ ]:
def _has_keyword(text):
    # Return True if text contains at least one election keyword
    t = text.lower()
    return any(kw in t for kw in ELECTION_KEYWORDS)


def search_posts(client, query, since=None, until=None):
    # Paginate through posts matching a hashtag; keep only keyword-matching posts
    collected, cursor = [], None
    while True:
        params = {"q": query, "limit": 100, "sort": "latest"}
        if cursor: params["cursor"] = cursor
        if since:  params["since"]  = since
        if until:  params["until"]  = until
        try:
            resp = client.app.bsky.feed.search_posts(params)
        except Exception as e:
            print(f"  API error: {e}"); break
        if not resp.posts: break
        for post in resp.posts:
            rec = post.record
            raw = getattr(rec, "text", "") or ""
            # Inline keyword filter
            if not _has_keyword(raw):
                continue
            text = re.sub(r"http\S+", "", raw)
            text = re.sub(r"\s+", " ", text).strip()
            collected.append({
                "uri"       : post.uri,
                "author"    : post.author.handle,
                "display"   : getattr(post.author, "display_name", "") or "",
                "text"      : text,
                "timestamp" : getattr(rec, "created_at", "") or "",
                "likes"     : post.like_count   or 0,
                "reposts"   : post.repost_count or 0,
                "replies"   : post.reply_count  or 0,
                "mentions"  : re.findall(r"@([\w.\-]+)", raw),
                "is_reply"  : bool(getattr(rec, "reply", None)),
                "post_type" : "post",
                "parent_uri": None,
                "query"     : query,
            })
        cursor = getattr(resp, "cursor", None)
        if not cursor: break
        time.sleep(0.5)
    return collected


def fetch_comments(client, uri, depth=3):
    # Fetch reply thread for a post; keep only keyword-matching comments
    try:
        resp = client.app.bsky.feed.get_post_thread({"uri": uri, "depth": depth})
    except Exception as e:
        print(f"  Thread error: {e}")
        return []

    comments = []

    def walk(node, parent_uri=None):
        if node is None:
            return
        post = getattr(node, "post", None)
        if post is None:
            return
        rec  = getattr(post, "record", None)
        raw  = getattr(rec, "text", "") or ""
        # Inline keyword filter
        if _has_keyword(raw):
            text = re.sub(r"http\S+", "", raw)
            text = re.sub(r"\s+", " ", text).strip()
            comments.append({
                "uri"       : post.uri,
                "author"    : post.author.handle,
                "display"   : getattr(post.author, "display_name", "") or "",
                "text"      : text,
                "timestamp" : getattr(rec, "created_at", "") or "",
                "likes"     : post.like_count   or 0,
                "reposts"   : post.repost_count or 0,
                "replies"   : post.reply_count  or 0,
                "mentions"  : re.findall(r"@([\w.\-]+)", raw),
                "is_reply"  : True,
                "post_type" : "comment",
                "parent_uri": parent_uri,
                "query"     : "thread",
            })
        for child in (getattr(node, "replies", None) or []):
            walk(child, parent_uri=post.uri)

    walk(getattr(resp, "thread", None))
    return comments


print("Functions ready: search_posts, fetch_comments")

## 3. Collect and Save

### bsky_US_2024_raw.csv columns

| Column | Type | Description |
|---|---|---|
| `uri` | str | Unique Bluesky post URI |
| `author` | str | Bluesky handle of the poster |
| `display` | str | Display name of the poster |
| `text` | str | Post text (URLs stripped) |
| `timestamp` | str | ISO 8601 creation time (UTC) |
| `likes` | int | Like count at collection time |
| `reposts` | int | Repost count at collection time |
| `replies` | int | Reply count at collection time |
| `mentions` | list | List of @-mentioned handles |
| `is_reply` | bool | True if this is a reply to another post |
| `post_type` | str | `post` (top-level) or `comment` (reply) |
| `query` | str | Hashtag query that retrieved the post, or `thread` for fetched comments |
| `parent_uri` | str | URI of the direct parent post/comment (`None` for top-level posts) |

In [ ]:
os.makedirs(os.path.dirname(os.path.abspath(BRONZE_CSV)), exist_ok=True)

# Step 1: Search posts per hashtag
all_posts = []
for i, tag in enumerate(HASHTAGS, 1):
    print(f"[{i}/{len(HASHTAGS)}] {tag} ...", end=" ", flush=True)
    posts = search_posts(client, tag, since=DATE_SINCE, until=DATE_UNTIL)
    all_posts.extend(posts)
    print(f"{len(posts)} posts  (total: {len(all_posts):,})")

# Deduplicate before fetching comments
df_posts = pd.DataFrame(all_posts).drop_duplicates(subset="uri").reset_index(drop=True)
print(f"\n{len(df_posts):,} unique posts — fetching comments...\n")

# Step 2: Fetch comments for each post
all_comments = []
for i, row in enumerate(df_posts.itertuples(), 1):
    if i % 100 == 0:
        print(f"  Comments: {i}/{len(df_posts)} posts processed  ({len(all_comments):,} so far)")
    all_comments.extend(fetch_comments(client, row.uri, depth=3))

print(f"\nComments collected: {len(all_comments):,}")

# Step 3: Merge, deduplicate and save to Bronze
df_raw = pd.concat([df_posts, pd.DataFrame(all_comments)], ignore_index=True)
df_raw = df_raw.drop_duplicates(subset="uri").reset_index(drop=True)
df_raw.to_csv(BRONZE_CSV, index=False)

print(f"\nBronze saved: {len(df_raw):,} unique rows  →  {BRONZE_CSV}")
print(f"Columns: {list(df_raw.columns)}")